# 🦄 РГР №2 по математическому анализу: Процедурный 3D-ландшафт на основе спектрального синтеза Фурье

**Тема проекта:** Генеративный процесс: Процедурный ландшафт
**Команда:** [Имена участников команды]

---

## 📖 Введение и теоретические основы

В компьютерной графике и симуляциях для моделирования природных ландшафтов (гор, холмов, долин) применяется **спектральный синтез**. Суть метода заключается в представлении рельефа как суперпозиции периодических колебаний (гармоник) различных частот и направлений. Математическим базисом этого подхода выступают **ряды и преобразования Фурье**.

### 1. Представление ландшафта через ряд Фурье
Двумерная поверхность ландшафта задается функцией высоты $Z(x, y)$ над плоскостью координат. Согласно теории Фурье, любая достаточно гладкая функция на ограниченной области может быть представлена в виде двойного ряда Фурье:

$$Z(x, y) = \sum_{n=-\infty}^{\infty} \sum_{m=-\infty}^{\infty} C_{nm} e^{i (k_x x + k_y y)}$$

где:
* $C_{nm}$ — комплексные амплитуды (спектральные коэффициенты),
* $ec{k} = (k_x, k_y) = \left(rac{2\pi n}{L_x}, rac{2\pi m}{L_y}ight)$ — волновой вектор частоты.

### 2. Спектральная плотность и фрактальный (розовый) шум
В реальной природе ландшафты обладают свойством самоподобия (фрактальности). Крупные формы рельефа (горы) имеют большую высоту (амплитуду), но встречаются редко (низкая частота). Мелкие детали (камни, трещины) имеют небольшую высоту, но встречаются часто (высокая частота).

Это описывается степенным законом спектральной плотности мощности (закон $1/f^eta$):

$$|C_{nm}| \propto rac{1}{|ec{k}|^{eta}}$$

где:
* $eta$ — **спектральный индекс** (показатель шероховатости ландшафта):
  * При $eta pprox 1.0 - 1.5$: амплитуда высоких частот затухает медленно, рельеф получается резким, скалистым и зубчатым (высокая фрактальная размерность).
  * При $eta pprox 2.0 - 2.5$: высокие частоты подавлены сильнее, ландшафт получается гладким, с пологими холмами.

---

## 🛠 Реализованные алгоритмы

В проекте реализованы два режима генерации:

### Метод А: Спектральный синтез через 2D FFT (Fast Fourier Transform)
Применяется для быстрого получения готовой карты высот фиксированного размера $N 	imes N$:
1. Генерируется матрица белого шума $W$.
2. Вычисляется прямое преобразование: $S = \text{fftshift}(\text{fft2}(W))$.
3. Применяется частотный фильтр спада амплитуд: $H(u, v) = S(u, v) \cdot rac{1}{(u^2 + v^2)^{eta/2}}$.
4. Выполняется обратное преобразование: $Z = \text{real}(\text{ifft2}(\text{ifftshift}(H)))$.
5. Карта высот нормализуется в диапазон $[0, 1]$.

### Метод Б: Аналитический метод гармоник
Используется для генерации бесконечного мира без швов. Высота рассчитывается аналитически для любой точки $(x, y)$:

$$Z(x, y) = \sum_{i=1}^{K} A_i \sin(k_{xi}x + k_{yi}y + \phi_i)$$

Амплитуда $A_i = rac{1}{|ec{k}_i|^eta}$, а направления $ec{k}_i$ и фазы $\phi_i$ выбираются псевдослучайно на основе начального зерна (*Seed*). Поскольку функция непрерывна, расчет соседних участков (чанков) гарантирует идеальное совпадение на стыках.


In [ ]:
# Импортируем необходимые библиотеки
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# Импортируем функции из нашего математического модуля
from landscape_synthesis import (
    generate_fft_terrain,
    generate_analytical_terrain,
    calculate_normals,
    calculate_lighting,
    get_biome_colors,
    get_shaded_terrain_colors
)

print("Все библиотеки успешно импортированы!")


## 🌊 Метод А: Быстрое Преобразование Фурье (2D FFT)

Ниже представлена интерактивная панель управления для генерации ландшафта методом 2D FFT.
Вы можете регулировать:
* **Размер сетки** (разрешение ландшафта).
* **Спектральный индекс $\beta$** (отвечает за фрактальную шероховатость рельефа).
* **Зерно (Seed)** для смены случайного рельефа.
* **Высоту и Азимут солнца** (для перемещения теней по склонам).
* **Масштаб высоты** (Z-scale) для регулирования перепада высот.
* **Режим биомов** (плавные переходы или дискретные зоны согласно ТЗ).


In [ ]:
# Определение интерактивной функции для FFT-генерации
@widgets.interact(
    size=widgets.IntSlider(min=64, max=256, step=64, value=128, description='Размер:'),
    beta=widgets.FloatSlider(min=0.5, max=3.0, step=0.1, value=1.8, description='Шероховатость β:'),
    seed=widgets.IntSlider(min=1, max=1000, step=1, value=42, description='Seed:'),
    sun_alt=widgets.FloatSlider(min=10, max=90, step=5, value=45, description='Высота солнца:'),
    sun_az=widgets.FloatSlider(min=0, max=360, step=10, value=135, description='Азимут солнца:'),
    z_scale=widgets.FloatSlider(min=5.0, max=50.0, step=2.5, value=25.0, description='Масштаб Z:'),
    smooth=widgets.Checkbox(value=True, description='Плавные биомы')
)
def interactive_fft_terrain(size, beta, seed, sun_alt, sun_az, z_scale, smooth):
    # Преобразуем углы солнца в декартов вектор направления
    alt_rad = np.radians(sun_alt)
    az_rad = np.radians(sun_az)
    lx = np.cos(alt_rad) * np.cos(az_rad)
    ly = np.cos(alt_rad) * np.sin(az_rad)
    lz = np.sin(alt_rad)
    light_dir = (lx, ly, lz)
    
    # 1. Генерируем массив высот через FFT
    Z = generate_fft_terrain(size=size, beta=beta, seed=seed)
    
    # 2. Получаем затененные цвета биомов
    colors = get_shaded_terrain_colors(Z, light_dir=light_dir, is_smooth=smooth)
    
    # 3. Переводим массив цветов в строки формата rgb для Plotly
    c_int = (colors * 255).astype(int)
    rgb_strings = np.char.add(
        np.char.add(
            np.char.add(
                np.char.add("rgb(", c_int[..., 0].astype(str)), 
                ","
            ), 
            np.char.add(c_int[..., 1].astype(str), ",")
        ),
        np.char.add(c_int[..., 2].astype(str), ")")
    )
    
    # 4. Создаем сетку координат
    x = np.arange(size)
    y = np.arange(size)
    X, Y = np.meshgrid(x, y)
    
    # 5. Строим 3D-поверхность в Plotly
    fig = go.Figure(data=[go.Surface(
        x=X, y=Y, z=Z * z_scale,
        surfacecolor=rgb_strings,
        showscale=False,
        hoverinfo='none'
    )])
    
    fig.update_layout(
        title=dict(text=f"Метод FFT (Размер: {size}x{size}, β={beta:.1f})", x=0.5),
        margin=dict(l=0, r=0, b=0, t=40),
        scene=dict(
            xaxis=dict(title='X', showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(title='Y', showgrid=False, zeroline=False, showticklabels=False),
            zaxis=dict(title='Height', showgrid=False, zeroline=False, showticklabels=False),
            aspectratio=dict(x=1, y=1, z=0.35)
        ),
        width=800,
        height=600
    )
    fig.show()


## ♾ Метод Б: Бесконечные бесшовные чанки (Аналитический метод)

Аналитический метод позволяет вычислить высоту в любой точке непрерывного пространства. Мы можем разбить мир на сетку квадратных областей («чанков») размера $M \times M$ и рендерить их независимо. Благодаря непрерывности исходных синусоид на стыках чанков не будет никаких швов и разрывов!

В данной интерактивной демонстрации вы можете перемещаться по миру с помощью слайдеров **Chunk Offset X** и **Chunk Offset Y**. Обратите внимание, что рельеф перетекает плавно из одного чанка в другой, сохраняя целостность холмов и впадин.


In [ ]:
# Определение интерактивной функции для бесконечных чанков
@widgets.interact(
    chunk_x=widgets.IntSlider(min=-5, max=5, step=1, value=0, description='Сдвиг X (Chunk):'),
    chunk_y=widgets.IntSlider(min=-5, max=5, step=1, value=0, description='Сдвиг Y (Chunk):'),
    beta=widgets.FloatSlider(min=0.5, max=3.0, step=0.1, value=1.8, description='Шероховатость β:'),
    num_harmonics=widgets.IntSlider(min=10, max=100, step=10, value=50, description='Гармоники (K):'),
    seed=widgets.IntSlider(min=1, max=1000, step=1, value=42, description='Seed:'),
    sun_alt=widgets.FloatSlider(min=10, max=90, step=5, value=45, description='Высота солнца:'),
    sun_az=widgets.FloatSlider(min=0, max=360, step=10, value=135, description='Азимут солнца:'),
    smooth=widgets.Checkbox(value=True, description='Плавные биомы')
)
def interactive_analytical_terrain(chunk_x, chunk_y, beta, num_harmonics, seed, sun_alt, sun_az, smooth):
    size = 100 # Размер сетки чанка
    chunk_size = 10.0 # Геометрический размер чанка в пространстве
    
    # 1. Формируем глобальные координаты чанка в зависимости от смещений
    x_start = chunk_x * chunk_size
    y_start = chunk_y * chunk_size
    
    x = np.linspace(x_start, x_start + chunk_size, size)
    y = np.linspace(y_start, y_start + chunk_size, size)
    X, Y = np.meshgrid(x, y)
    
    # 2. Генерируем карту высот аналитически
    Z = generate_analytical_terrain(X, Y, beta=beta, num_harmonics=num_harmonics, seed=seed)
    
    # 3. Вычисляем свет
    alt_rad = np.radians(sun_alt)
    az_rad = np.radians(sun_az)
    lx = np.cos(alt_rad) * np.cos(az_rad)
    ly = np.cos(alt_rad) * np.sin(az_rad)
    lz = np.sin(alt_rad)
    light_dir = (lx, ly, lz)
    
    # Считаем шаг сетки для правильного расчета нормалей
    dx = chunk_size / (size - 1)
    dy = chunk_size / (size - 1)
    normals = calculate_normals(Z, dx=dx, dy=dy)
    intensity = calculate_lighting(normals, light_dir=light_dir)
    colors = get_shaded_terrain_colors(Z, light_dir=light_dir, is_smooth=smooth)
    
    # 4. Цвета в rgb строки
    c_int = (colors * 255).astype(int)
    rgb_strings = np.char.add(
        np.char.add(
            np.char.add(
                np.char.add("rgb(", c_int[..., 0].astype(str)), 
                ","
            ), 
            np.char.add(c_int[..., 1].astype(str), ",")
        ),
        np.char.add(c_int[..., 2].astype(str), ")")
    )
    
    # 5. Отрисовка
    fig = go.Figure(data=[go.Surface(
        x=X, y=Y, z=Z * 3.0, # Масштабируем высоту для визуальной красоты
        surfacecolor=rgb_strings,
        showscale=False,
        hoverinfo='none'
    )])
    
    fig.update_layout(
        title=dict(text=f"Аналитический Чанк ({chunk_x}, {chunk_y}) | K={num_harmonics}, β={beta:.1f}", x=0.5),
        margin=dict(l=0, r=0, b=0, t=40),
        scene=dict(
            xaxis=dict(title='X (Глобальная)', showgrid=True),
            yaxis=dict(title='Y (Глобальная)', showgrid=True),
            zaxis=dict(title='Z', showgrid=False, showticklabels=False),
            aspectratio=dict(x=1, y=1, z=0.35)
        ),
        width=800,
        height=600
    )
    fig.show()


## 📝 Заключение

В ходе работы над РГР2 были решены следующие задачи:
1. **Связь теории с практикой:** Продемонстрировано, как спектральное разложение Фурье (ослабление амплитуд высоких частот по степенному закону $1/f^\beta$) порождает фрактальные текстуры, аналогичные природному рельефу.
2. **Алгоритмическая реализация:**
   - Реализована фильтрация шума в частотной плоскости с помощью **2D Fast Fourier Transform** (метод FFT).
   - Реализована **аналитическая функция рельефа** на основе суммирования гармоник, обеспечивающая непрерывность и бесшовную генерацию миров.
3. **Физическая визуализация:**
   - Написан расчет частных производных высот методом конечных разностей для определения нормалей.
   - Реализована светотеневая модель Ламберта для отображения честных трехмерных теней на склонах.
   - Реализовано процедурное текстурирование по высоте (карта биомов) для реалистичного отображения воды, песка, равнин, скал и снега.
